In [4]:
import pandas as pd
import numpy as np
import sqlite3
import warnings

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    KFold,
    cross_validate
)

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder,
    LabelEncoder
)

from sklearn.impute import SimpleImputer

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression, LinearRegression

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    GradientBoostingRegressor
)

from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    confusion_matrix,
    classification_report
)

from xgboost import XGBClassifier, XGBRegressor

warnings.filterwarnings("ignore")

print("Stage 7 libraries imported successfully.")

Stage 7 libraries imported successfully.


In [6]:
import sqlite3
# ============================================================
# STAGE 6.2 — LOAD DATA FROM SQL DATABASE
# ============================================================
db_path = r"C:\Users\ADMIN\smart agricultural analytics\agriculture.db"
conn = sqlite3.connect(db_path)

# Check available tables
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table';
    """,
    conn
)

print("Available tables:")
display(tables)

Available tables:


,name
0,crop_recommendation
1,agri_yield_train
2,agri_yield_test
3,crop_recommendation_clean
4,agri_yield_train_clean
5,agri_yield_test_clean
6,crop_recommendation_fe
7,agri_yield_train_fe
8,agri_yield_test_fe
9,crop_recommendation_selected


In [7]:
sf24 = pd.read_sql_query(
    "SELECT * FROM crop_recommendation_selected",
    conn
)

agri_train = pd.read_sql_query(
    "SELECT * FROM agri_yield_train_selected",
    conn
)

agri_test = pd.read_sql_query(
    "SELECT * FROM agri_yield_test_selected",
    conn
)

print("Datasets loaded successfully.")

print("\nSF24 shape:", sf24.shape)
print("AgriYield train shape:", agri_train.shape)
print("AgriYield test shape:", agri_test.shape)

Datasets loaded successfully.

SF24 shape: (2200, 13)
AgriYield train shape: (10000, 11)
AgriYield test shape: (2000, 10)


In [8]:
print("SF24 shape:", sf24.shape)
print("AgriYield train shape:", agri_train.shape)
print("AgriYield test shape:", agri_test.shape)

print("\nSF24 columns:")
print(sf24.columns.tolist())

print("\nAgriYield train columns:")
print(agri_train.columns.tolist())

print("\nAgriYield test columns:")
print(agri_test.columns.tolist())

SF24 shape: (2200, 13)
AgriYield train shape: (10000, 11)
AgriYield test shape: (2000, 10)

SF24 columns:
['humidity', 'K', 'temperature', 'N', 'P_K_Ratio', 'NPK_Total', 'Temperature_Rainfall', 'rainfall', 'N_K_Ratio', 'Humidity_Rainfall', 'Temperature_Humidity', 'P', 'label']

AgriYield train columns:
['NDVI_Temperature', 'OrganicMatter_Sand', 'Temperature_Rainfall', 'sand_pct', 'NDVI_Rainfall', 'soil_ph', 'organic_matter', 'Humidity_Rainfall', 'Temperature_Humidity', 'rainfall', 'yield']

AgriYield test columns:
['NDVI_Temperature', 'OrganicMatter_Sand', 'Temperature_Rainfall', 'sand_pct', 'NDVI_Rainfall', 'soil_ph', 'organic_matter', 'Humidity_Rainfall', 'Temperature_Humidity', 'rainfall']


In [9]:
SF24_TARGET = "label"
AGRI_TARGET = "yield"

# SF24 classification
X_sf24 = sf24.drop(columns=[SF24_TARGET])
y_sf24 = sf24[SF24_TARGET]

# AgriYield regression
X_agri = agri_train.drop(columns=[AGRI_TARGET])
y_agri = agri_train[AGRI_TARGET]

# AgriYield test features
X_agri_test = agri_test.copy()

print("SF24 X:", X_sf24.shape)
print("SF24 y:", y_sf24.shape)

print("\nAgriYield X:", X_agri.shape)
print("AgriYield y:", y_agri.shape)

print("\nAgriYield test X:", X_agri_test.shape)

SF24 X: (2200, 12)
SF24 y: (2200,)

AgriYield X: (10000, 10)
AgriYield y: (10000,)

AgriYield test X: (2000, 10)


In [10]:
print("SF24 target datatype:")
print(y_sf24.dtype)

print("\nNumber of classes:")
print(y_sf24.nunique())

print("\nClasses:")
print(y_sf24.unique())

SF24 target datatype:
object

Number of classes:
22

Classes:
['rice' 'maize' 'chickpea' 'kidneybeans' 'pigeonpeas' 'mothbeans'
 'mungbean' 'blackgram' 'lentil' 'pomegranate' 'banana' 'mango' 'grapes'
 'watermelon' 'muskmelon' 'apple' 'orange' 'papaya' 'coconut' 'cotton'
 'jute' 'coffee']


In [11]:
from sklearn.preprocessing import LabelEncoder

sf24_label_encoder = LabelEncoder()

y_sf24_encoded = sf24_label_encoder.fit_transform(y_sf24)

print("Class mapping:")

for number, crop in enumerate(sf24_label_encoder.classes_):
    print(number, "->", crop)

Class mapping:
0 -> apple
1 -> banana
2 -> blackgram
3 -> chickpea
4 -> coconut
5 -> coffee
6 -> cotton
7 -> grapes
8 -> jute
9 -> kidneybeans
10 -> lentil
11 -> maize
12 -> mango
13 -> mothbeans
14 -> mungbean
15 -> muskmelon
16 -> orange
17 -> papaya
18 -> pigeonpeas
19 -> pomegranate
20 -> rice
21 -> watermelon


In [12]:
from sklearn.model_selection import train_test_split

X_sf24_train, X_sf24_test, y_sf24_train, y_sf24_test = train_test_split(
    X_sf24,
    y_sf24_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_sf24_encoded
)

print("SF24 training data:", X_sf24_train.shape)
print("SF24 holdout data:", X_sf24_test.shape)

print("\nTraining classes:")
print(pd.Series(y_sf24_train).value_counts().sort_index())

print("\nHoldout classes:")
print(pd.Series(y_sf24_test).value_counts().sort_index())

SF24 training data: (1760, 12)
SF24 holdout data: (440, 12)

Training classes:
0     80
1     80
2     80
3     80
4     80
5     80
6     80
7     80
8     80
9     80
10    80
11    80
12    80
13    80
14    80
15    80
16    80
17    80
18    80
19    80
20    80
21    80
Name: count, dtype: int64

Holdout classes:
0     20
1     20
2     20
3     20
4     20
5     20
6     20
7     20
8     20
9     20
10    20
11    20
12    20
13    20
14    20
15    20
16    20
17    20
18    20
19    20
20    20
21    20
Name: count, dtype: int64


In [13]:
sf24_numeric_features = X_sf24_train.select_dtypes(
    include=np.number
).columns.tolist()

sf24_categorical_features = X_sf24_train.select_dtypes(
    exclude=np.number
).columns.tolist()

agri_numeric_features = X_agri.select_dtypes(
    include=np.number
).columns.tolist()

agri_categorical_features = X_agri.select_dtypes(
    exclude=np.number
).columns.tolist()

print("SF24 numerical features:")
print(sf24_numeric_features)

print("\nSF24 categorical features:")
print(sf24_categorical_features)

print("\nAgriYield numerical features:")
print(agri_numeric_features)

print("\nAgriYield categorical features:")
print(agri_categorical_features)

SF24 numerical features:
['humidity', 'K', 'temperature', 'N', 'P_K_Ratio', 'NPK_Total', 'Temperature_Rainfall', 'rainfall', 'N_K_Ratio', 'Humidity_Rainfall', 'Temperature_Humidity', 'P']

SF24 categorical features:
[]

AgriYield numerical features:
['NDVI_Temperature', 'OrganicMatter_Sand', 'Temperature_Rainfall', 'sand_pct', 'NDVI_Rainfall', 'soil_ph', 'organic_matter', 'Humidity_Rainfall', 'Temperature_Humidity', 'rainfall']

AgriYield categorical features:
[]


In [14]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

sf24_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

sf24_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

sf24_preprocessor = ColumnTransformer([
    (
        "numeric",
        sf24_numeric_pipeline,
        sf24_numeric_features
    ),
    (
        "categorical",
        sf24_categorical_pipeline,
        sf24_categorical_features
    )
])

print("SF24 preprocessing pipeline created.")

SF24 preprocessing pipeline created.


In [15]:
agri_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

agri_categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

agri_preprocessor = ColumnTransformer([
    (
        "numeric",
        agri_numeric_pipeline,
        agri_numeric_features
    ),
    (
        "categorical",
        agri_categorical_pipeline,
        agri_categorical_features
    )
])

print("AgriYield preprocessing pipeline created.")

AgriYield preprocessing pipeline created.


In [16]:
from sklearn.model_selection import StratifiedKFold, KFold

N_SPLITS = 5
RANDOM_STATE = 42

sf24_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

agri_cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("SF24: 5-fold Stratified CV")
print("AgriYield: 5-fold KFold CV")

SF24: 5-fold Stratified CV
AgriYield: 5-fold KFold CV


In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

sf24_models = {

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "SVM": SVC(
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="multi:softprob",
        num_class=len(sf24_label_encoder.classes_),
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    )
}

print("SF24 models created:")
for model in sf24_models:
    print("-", model)

SF24 models created:
- Logistic Regression
- Decision Tree
- Random Forest
- SVM
- XGBoost


In [18]:
from sklearn.model_selection import cross_validate

sf24_scoring = {
    "Accuracy": "accuracy",
    "Precision": "precision_weighted",
    "Recall": "recall_weighted",
    "F1": "f1_weighted"
}

sf24_cv_results = []

for model_name, model in sf24_models.items():

    pipeline = Pipeline([
        ("preprocessor", sf24_preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X_sf24_train,
        y_sf24_train,
        cv=sf24_cv,
        scoring=sf24_scoring,
        n_jobs=-1,
        return_train_score=False
    )

    sf24_cv_results.append({
        "Model": model_name,

        "Accuracy Mean": scores["test_Accuracy"].mean(),
        "Accuracy Std": scores["test_Accuracy"].std(),

        "Precision Mean": scores["test_Precision"].mean(),
        "Precision Std": scores["test_Precision"].std(),

        "Recall Mean": scores["test_Recall"].mean(),
        "Recall Std": scores["test_Recall"].std(),

        "F1 Mean": scores["test_F1"].mean(),
        "F1 Std": scores["test_F1"].std()
    })

sf24_cv_results_df = pd.DataFrame(sf24_cv_results)

sf24_cv_results_df = sf24_cv_results_df.sort_values(
    by="F1 Mean",
    ascending=False
).reset_index(drop=True)

sf24_cv_results_df

,Model,Accuracy Mean,Accuracy Std,Precision Mean,Precision Std,Recall Mean,Recall Std,F1 Mean,F1 Std
0,Random Forest,0.989205,0.006067,0.990208,0.005275,0.989205,0.006067,0.989073,0.006226
1,XGBoost,0.980114,0.010322,0.981175,0.009846,0.980114,0.010322,0.980073,0.010400
2,Decision Tree,0.978977,0.006865,0.980454,0.005083,0.978977,0.006865,0.979018,0.006654
3,SVM,0.976136,0.011449,0.979766,0.008696,0.976136,0.011449,0.975790,0.012183
4,Logistic Regression,0.970455,0.006626,0.973240,0.005745,0.970455,0.006626,0.970316,0.006663


In [19]:
from pathlib import Path

stage7_output_dir = Path("stage7_outputs")
stage7_output_dir.mkdir(exist_ok=True)

sf24_cv_results_df.to_csv(
    stage7_output_dir / "sf24_model_comparison.csv",
    index=False
)

print("Saved:", stage7_output_dir / "sf24_model_comparison.csv")

Saved: stage7_outputs\sf24_model_comparison.csv


In [20]:
best_sf24_model_name = sf24_cv_results_df.iloc[0]["Model"]

print("Best SF24 model based on weighted F1:")
print(best_sf24_model_name)

Best SF24 model based on weighted F1:
Random Forest


In [21]:
best_sf24_model = sf24_models[best_sf24_model_name]

best_sf24_pipeline = Pipeline([
    ("preprocessor", sf24_preprocessor),
    ("model", best_sf24_model)
])

best_sf24_pipeline.fit(
    X_sf24_train,
    y_sf24_train
)

print("Best SF24 model trained successfully.")

Best SF24 model trained successfully.


In [22]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

sf24_test_predictions = best_sf24_pipeline.predict(
    X_sf24_test
)

sf24_test_accuracy = accuracy_score(
    y_sf24_test,
    sf24_test_predictions
)

sf24_test_precision = precision_score(
    y_sf24_test,
    sf24_test_predictions,
    average="weighted"
)

sf24_test_recall = recall_score(
    y_sf24_test,
    sf24_test_predictions,
    average="weighted"
)

sf24_test_f1 = f1_score(
    y_sf24_test,
    sf24_test_predictions,
    average="weighted"
)

print("SF24 HOLDOUT RESULTS")
print("=" * 40)

print("Model    :", best_sf24_model_name)
print("Accuracy :", sf24_test_accuracy)
print("Precision:", sf24_test_precision)
print("Recall   :", sf24_test_recall)
print("F1       :", sf24_test_f1)

SF24 HOLDOUT RESULTS
Model    : Random Forest
Accuracy : 0.9931818181818182
Precision: 0.9935064935064934
Recall   : 0.9931818181818182
F1       : 0.9931775541531639


In [23]:
from sklearn.metrics import confusion_matrix

sf24_cm = confusion_matrix(
    y_sf24_test,
    sf24_test_predictions
)

sf24_cm_df = pd.DataFrame(
    sf24_cm,
    index=sf24_label_encoder.classes_,
    columns=sf24_label_encoder.classes_
)

sf24_cm_df

,apple,banana,blackgram,chickpea,coconut,coffee,cotton,grapes,jute,kidneybeans,...,mango,mothbeans,mungbean,muskmelon,orange,papaya,pigeonpeas,pomegranate,rice,watermelon
apple,20,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
banana,0,20,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
blackgram,0,0,19,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
chickpea,0,0,0,20,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
coconut,0,0,0,0,20,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
coffee,0,0,0,0,0,20,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
cotton,0,0,0,0,0,0,20,0,0,0,...,0,0,0,0,0,0,0,0,0,0
grapes,0,0,0,0,0,0,0,20,0,0,...,0,0,0,0,0,0,0,0,0,0
jute,0,0,0,0,0,0,0,0,20,0,...,0,0,0,0,0,0,0,0,0,0
kidneybeans,0,0,0,0,0,0,0,0,0,20,...,0,0,0,0,0,0,0,0,0,0


In [24]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_sf24_test,
        sf24_test_predictions,
        target_names=sf24_label_encoder.classes_
    )
)

              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       1.00      0.95      0.97        20
    chickpea       1.00      1.00      1.00        20
     coconut       1.00      1.00      1.00        20
      coffee       1.00      1.00      1.00        20
      cotton       1.00      1.00      1.00        20
      grapes       1.00      1.00      1.00        20
        jute       0.95      1.00      0.98        20
 kidneybeans       1.00      1.00      1.00        20
      lentil       1.00      0.95      0.97        20
       maize       0.95      1.00      0.98        20
       mango       1.00      1.00      1.00        20
   mothbeans       0.95      1.00      0.98        20
    mungbean       1.00      1.00      1.00        20
   muskmelon       1.00      1.00      1.00        20
      orange       1.00      1.00      1.00        20
      papaya       1.00    

In [25]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

agri_models = {

    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
}

print("AgriYield regression models created:")

for model_name in agri_models:
    print("-", model_name)

AgriYield regression models created:
- Linear Regression
- Random Forest
- Gradient Boosting
- XGBoost


In [26]:
agri_scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

print("Regression scoring metrics defined.")

Regression scoring metrics defined.


In [27]:

agri_cv_results = []

for model_name, model in agri_models.items():

    pipeline = Pipeline([
        ("preprocessor", agri_preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipeline,
        X_agri,
        y_agri,
        cv=agri_cv,
        scoring=agri_scoring,
        n_jobs=-1,
        return_train_score=False
    )

    mae_scores = -scores["test_MAE"]
    rmse_scores = -scores["test_RMSE"]
    r2_scores = scores["test_R2"]

    agri_cv_results.append({

        "Model": model_name,

        "MAE Mean": mae_scores.mean(),
        "MAE Std": mae_scores.std(),

        "RMSE Mean": rmse_scores.mean(),
        "RMSE Std": rmse_scores.std(),

        "R2 Mean": r2_scores.mean(),
        "R2 Std": r2_scores.std()
    })

agri_cv_results_df = pd.DataFrame(agri_cv_results)

agri_cv_results_df = agri_cv_results_df.sort_values(
    by="R2 Mean",
    ascending=False
).reset_index(drop=True)

agri_cv_results_df

,Model,MAE Mean,MAE Std,RMSE Mean,RMSE Std,R2 Mean,R2 Std
0,Linear Regression,378.750282,5.435120,436.661181,5.106936,-0.001094,0.000927
1,Gradient Boosting,381.570824,6.085259,441.075143,5.986048,-0.021414,0.007455
2,Random Forest,384.533580,6.123718,445.447543,6.280316,-0.041791,0.011885
3,XGBoost,389.417378,4.433518,454.227373,4.158108,-0.083363,0.010026


In [28]:
agri_cv_results_df.to_csv(
    stage7_output_dir / "agriyield_model_comparison.csv",
    index=False
)

print(
    "Saved:",
    stage7_output_dir / "agriyield_model_comparison.csv"
)

Saved: stage7_outputs\agriyield_model_comparison.csv


In [29]:
best_agri_model_name = agri_cv_results_df.iloc[0]["Model"]

print("Best AgriYield model based on CV R²:")
print(best_agri_model_name)

Best AgriYield model based on CV R²:
Linear Regression


In [30]:
best_agri_model = agri_models[best_agri_model_name]

best_agri_pipeline = Pipeline([
    ("preprocessor", agri_preprocessor),
    ("model", best_agri_model)
])

best_agri_pipeline.fit(
    X_agri,
    y_agri
)

print("Best AgriYield model trained successfully.")

Best AgriYield model trained successfully.


In [31]:
agri_test_predictions = best_agri_pipeline.predict(
    X_agri_test
)

agri_test_predictions_df = pd.DataFrame({
    "predicted_yield": agri_test_predictions
})

print("Predictions generated successfully.")

print("\nFirst 10 predictions:")
print(agri_test_predictions_df.head(10))

Predictions generated successfully.

First 10 predictions:
   predicted_yield
0      4259.350699
1      4288.650345
2      4238.443570
3      4229.374547
4      4255.917044
5      4233.837025
6      4248.369040
7      4276.245619
8      4258.729995
9      4257.553708


In [32]:
agri_test_predictions_df.to_csv(
    stage7_output_dir / "agriyield_best_model_test_predictions.csv",
    index=False
)

print(
    "Saved:",
    stage7_output_dir / "agriyield_best_model_test_predictions.csv"
)


Saved: stage7_outputs\agriyield_best_model_test_predictions.csv


In [33]:
sf24_final_results = pd.DataFrame({
    "Model": [best_sf24_model_name],
    "Accuracy": [sf24_test_accuracy],
    "Precision": [sf24_test_precision],
    "Recall": [sf24_test_recall],
    "F1": [sf24_test_f1]
})

sf24_final_results

,Model,Accuracy,Precision,Recall,F1
0,Random Forest,0.993182,0.993506,0.993182,0.993178


In [34]:
sf24_final_results.to_csv(
    stage7_output_dir / "sf24_best_model_test_results.csv",
    index=False
)

sf24_cm_df.to_csv(
    stage7_output_dir / "sf24_confusion_matrix.csv"
)

print("SF24 final results saved.")

SF24 final results saved.


In [35]:
stage7_summary = pd.DataFrame({

    "Task": [
        "SF24 Classification",
        "AgriYield Regression"
    ],

    "Best Model": [
        best_sf24_model_name,
        best_agri_model_name
    ],

    "Primary Metric": [
        "Weighted F1",
        "R²"
    ],

    "Validation Strategy": [
        "5-Fold Stratified CV + 20% Holdout",
        "5-Fold KFold CV"
    ]
})

stage7_summary

,Task,Best Model,Primary Metric,Validation Strategy
0,SF24 Classification,Random Forest,Weighted F1,5-Fold Stratified CV + 20% Holdout
1,AgriYield Regression,Linear Regression,R²,5-Fold KFold CV


In [36]:
stage7_summary.to_csv(
    stage7_output_dir / "stage7_summary.csv",
    index=False
)

print("Stage 7 summary saved.")

Stage 7 summary saved.


In [37]:
print("=" * 70)
print("STAGE 7 — BASELINE MODEL DEVELOPMENT & COMPARISON")
print("=" * 70)

print("\nSF24 CLASSIFICATION")
print("-" * 70)

print("Best model:", best_sf24_model_name)

print("Holdout Accuracy :",
      round(sf24_test_accuracy, 4))

print("Holdout Precision:",
      round(sf24_test_precision, 4))

print("Holdout Recall   :",
      round(sf24_test_recall, 4))

print("Holdout F1       :",
      round(sf24_test_f1, 4))


print("\nSF24 MODEL COMPARISON")
print("-" * 70)

print(
    sf24_cv_results_df.to_string(index=False)
)


print("\nAGRIYIELD REGRESSION")
print("-" * 70)

print("Best model:", best_agri_model_name)


print("\nAGRIYIELD MODEL COMPARISON")
print("-" * 70)

print(
    agri_cv_results_df.to_string(index=False)
)


print("\nOUTPUT FILES")
print("-" * 70)

print("Output directory:", stage7_output_dir)

print("\nStage 7 completed successfully.")

STAGE 7 — BASELINE MODEL DEVELOPMENT & COMPARISON

SF24 CLASSIFICATION
----------------------------------------------------------------------
Best model: Random Forest
Holdout Accuracy : 0.9932
Holdout Precision: 0.9935
Holdout Recall   : 0.9932
Holdout F1       : 0.9932

SF24 MODEL COMPARISON
----------------------------------------------------------------------
              Model  Accuracy Mean  Accuracy Std  Precision Mean  Precision Std  Recall Mean  Recall Std  F1 Mean   F1 Std
      Random Forest       0.989205      0.006067        0.990208       0.005275     0.989205    0.006067 0.989073 0.006226
            XGBoost       0.980114      0.010322        0.981175       0.009846     0.980114    0.010322 0.980073 0.010400
      Decision Tree       0.978977      0.006865        0.980454       0.005083     0.978977    0.006865 0.979018 0.006654
                SVM       0.976136      0.011449        0.979766       0.008696     0.976136    0.011449 0.975790 0.012183
Logistic Regression